<a href="https://colab.research.google.com/github/ivansuarez-AU/EGN321_Module7_Assignment7_2/blob/main/EGN321_Module7_Assignment7_2_COMPLETE_INSTRUCTOR_DEMO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EGN 321 — Module 7 Assignment 7.2 Complete Instructor Demo
## Failure Investigation, Recovery, Ollama Review, and Report Export

This notebook is designed for the **instructor to demo the entire Assignment 7.2 workflow** in class.

It creates a controlled failure, analyzes structured logs, builds the required evidence table, identifies the first observable symptom, proposes two causes, runs deterministic verification tests, performs recovery, uses Ollama as an evidence reviewer, generates the required failure report, and exports all supporting documents in one ZIP.

### Required report sections produced
1. System Before Failure  
2. Failure Introduced  
3. First Observable Symptom  
4. Log Evidence  
5. Timeline of Events  
6. Possible Cause 1  
7. Possible Cause 2  
8. Recovery Procedure  
9. Verification of Recovery  
10. What the Logs Told Me  
11. What the Logs Did Not Tell Me  
12. Improvement Made  
13. Remaining Limitations


## Demo Flow

```text
Healthy System
    ↓
Controlled Failure
    ↓
Structured Logs
    ↓
Required Evidence Table
    ↓
First Observable Symptom
    ↓
Possible Causes
    ↓
Deterministic Verification Tests
    ↓
Recovery
    ↓
Recovery Verification
    ↓
Ollama Review / Red Team
    ↓
Failure Report
    ↓
README + AI_LOG + ZIP
```

**Important:** logs and tests establish evidence. Ollama reviews, challenges, and helps draft from that evidence.


In [1]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pandas as pd
import sqlite3
import json
import shutil
import zipfile
import os

BASE = Path("/content/module7_assignment7_2_instructor_demo")
BASE.mkdir(exist_ok=True)

LOG_FILE = BASE / "module7_app.jsonl"
DB_FILE = BASE / "module7_demo.db"
OFFLINE_DB = BASE / "module7_demo.db.offline"
EXPORT_DIR = BASE / "assignment7_2_export"
EXPORT_DIR.mkdir(exist_ok=True)

for p in [LOG_FILE, DB_FILE, OFFLINE_DB]:
    if p.exists():
        p.unlink()

def log_event(level, event, component, **fields):
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "component": component,
        "event": event,
        **fields
    }
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record

print("Demo workspace:", BASE)


Demo workspace: /content/module7_assignment7_2_instructor_demo


## 1. System Before Failure

Create a healthy SQLite database, store readings, and record normal operational logs.


In [2]:
conn = sqlite3.connect(DB_FILE)

conn.execute("""
CREATE TABLE readings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    temperature_c REAL,
    status TEXT
)
""")

base_time = datetime.now(timezone.utc)

for i in range(1, 21):
    ts = (base_time + timedelta(seconds=i*2)).isoformat()
    value = 23.0 + (i * 0.03)

    conn.execute(
        "INSERT INTO readings(timestamp, temperature_c, status) VALUES(?,?,?)",
        (ts, value, "OK")
    )

    log_event(
        "INFO",
        "reading_stored",
        "sensor",
        sample_number=i,
        temperature_c=round(value, 3),
        sensor_status="OK",
        health_state="HEALTHY"
    )

conn.commit()
conn.close()

log_event(
    "INFO",
    "system_baseline",
    "system",
    database_available=True,
    dashboard_available=True,
    sensor_available=True,
    health_state="HEALTHY"
)

print("Healthy baseline created.")


Healthy baseline created.


## 2. Failure Introduced

For the demo, the database file is renamed. This simulates a realistic **database unavailable / wrong path** failure while keeping the original database recoverable.


In [3]:
shutil.move(DB_FILE, OFFLINE_DB)

log_event(
    "WARNING",
    "failure_injected",
    "storage",
    failure_type="database_unavailable",
    action="database file renamed",
    expected_effect="application cannot access expected readings table",
    health_state="DEGRADED"
)

try:
    conn = sqlite3.connect(DB_FILE)

    # SQLite may create an empty file automatically, so force a query
    # that requires the expected application schema.
    conn.execute("SELECT COUNT(*) FROM readings").fetchone()
    conn.close()

except Exception as exc:
    log_event(
        "ERROR",
        "database_read_failed",
        "storage",
        database_path=str(DB_FILE),
        exception_type=type(exc).__name__,
        exception_message=str(exc),
        health_state="FAILED"
    )

print("Controlled database failure introduced.")


Controlled database failure introduced.


## 3. Load and Parse the Logs

This is where the investigation starts. The report should be built from evidence, not from memory.


In [4]:
records = []

with open(LOG_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

logs = pd.DataFrame(records)
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True)
logs = logs.sort_values("timestamp").reset_index(drop=True)

print("Log rows:", len(logs))
print("Time range:", logs["timestamp"].min(), "to", logs["timestamp"].max())

display(
    logs[["timestamp","level","component","event","health_state"]]
    .tail(12)
)


Log rows: 23
Time range: 2026-10-15 20:09:05.704905+00:00 to 2026-10-15 20:09:17.570786+00:00


,timestamp,level,component,event,health_state
11,2026-10-15 20:09:05.709055+00:00,INFO,sensor,reading_stored,HEALTHY
12,2026-10-15 20:09:05.709137+00:00,INFO,sensor,reading_stored,HEALTHY
13,2026-10-15 20:09:05.709209+00:00,INFO,sensor,reading_stored,HEALTHY
14,2026-10-15 20:09:05.709279+00:00,INFO,sensor,reading_stored,HEALTHY
15,2026-10-15 20:09:05.709347+00:00,INFO,sensor,reading_stored,HEALTHY
16,2026-10-15 20:09:05.709415+00:00,INFO,sensor,reading_stored,HEALTHY
17,2026-10-15 20:09:05.709483+00:00,INFO,sensor,reading_stored,HEALTHY
18,2026-10-15 20:09:05.709553+00:00,INFO,sensor,reading_stored,HEALTHY
19,2026-10-15 20:09:05.709618+00:00,INFO,sensor,reading_stored,HEALTHY
20,2026-10-15 20:09:05.722409+00:00,INFO,system,system_baseline,HEALTHY


## 4. Required Evidence Table

The assignment requires students to convert logs into a simple evidence table:

| Time | Component | Event | Evidence | Interpretation |
|---|---|---|---|---|

This forces the investigation to separate **what happened** from **what the student thinks it means**.


In [5]:
def interpretation_for_event(row):
    event = row["event"]

    if event == "system_baseline":
        return "System confirmed healthy before the controlled failure."
    if event == "failure_injected":
        return "Expected database path was intentionally made unavailable."
    if event == "database_read_failed":
        return "Application could not access the expected readings table."
    if event == "recovery_action":
        return "Recovery action attempted to restore expected database path."
    if event == "recovery_verified":
        return "Database, schema, and preserved data were verified after recovery."
    if event == "logging_improvement_added":
        return "Additional evidence fields were added for future investigations."

    return "Operational event; review context before drawing a conclusion."

evidence_table = logs.copy()

evidence_table["Time"] = evidence_table["timestamp"].dt.strftime("%H:%M:%S")
evidence_table["Component"] = evidence_table["component"]
evidence_table["Event"] = evidence_table["event"]
evidence_table["Evidence"] = evidence_table["level"].map(
    lambda x: f"{x} log entry"
)
evidence_table["Interpretation"] = evidence_table.apply(
    interpretation_for_event,
    axis=1
)

required_evidence = evidence_table[
    ["Time","Component","Event","Evidence","Interpretation"]
].copy()

display(required_evidence.tail(10))


,Time,Component,Event,Evidence,Interpretation
13,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
14,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
15,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
16,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
17,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
18,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
19,20:09:05,sensor,reading_stored,INFO log entry,Operational event; review context before drawi...
20,20:09:05,system,system_baseline,INFO log entry,System confirmed healthy before the controlled...
21,20:09:17,storage,failure_injected,WARNING log entry,Expected database path was intentionally made ...
22,20:09:17,storage,database_read_failed,ERROR log entry,Application could not access the expected read...


## 5. First Observable Symptom

Do not automatically call the first logged symptom the root cause.

The correct wording is:

> **The first observable symptom in the available evidence was...**

That keeps the report evidence-based.


In [6]:
problem_logs = logs[
    logs["level"].isin(["WARNING","ERROR","ALERT"])
].copy()

first_problem = problem_logs.iloc[0]

print("First observable warning/error:")
display(
    first_problem[
        ["timestamp","level","component","event","health_state"]
    ].to_frame("value")
)

FIRST_OBSERVABLE = first_problem["event"]

print("\nReport wording:")
print(
    f"The first observable symptom in the available logs was "
    f"'{FIRST_OBSERVABLE}'. This does not by itself prove the final root cause."
)


First observable warning/error:


,value
timestamp,2026-10-15 20:09:17.569684+00:00
level,WARNING
component,storage
event,failure_injected
health_state,DEGRADED



Report wording:
The first observable symptom in the available logs was 'failure_injected'. This does not by itself prove the final root cause.


## 6. Build the Failure Timeline


In [7]:
timeline_events = [
    "system_baseline",
    "failure_injected",
    "database_read_failed"
]

failure_timeline = logs[
    logs["event"].isin(timeline_events)
][
    ["timestamp","level","component","event","health_state"]
].copy()

display(failure_timeline)


,timestamp,level,component,event,health_state
20,2026-10-15 20:09:05.722409+00:00,INFO,system,system_baseline,HEALTHY
21,2026-10-15 20:09:17.569684+00:00,WARNING,storage,failure_injected,DEGRADED
22,2026-10-15 20:09:17.570786+00:00,ERROR,storage,database_read_failed,FAILED


## 7. Possible Cause 1 and Verification Test

### Possible Cause 1
The expected database path became unavailable or changed.

### Verification Test
Check whether the expected database path exists.


In [8]:
def verify_database_path(path):
    p = Path(path)

    return {
        "test": "database_path_exists",
        "database_path": str(p),
        "exists": p.exists()
    }

cause1_before = verify_database_path(DB_FILE)

print(json.dumps(cause1_before, indent=2))


{
  "test": "database_path_exists",
  "database_path": "/content/module7_assignment7_2_instructor_demo/module7_demo.db",
  "exists": true
}


## 8. Possible Cause 2 and Verification Test

### Possible Cause 2
A database file exists, but the expected `readings` table is missing or invalid.

### Verification Test
Inspect SQLite metadata and confirm that the required table exists.


In [9]:
def verify_database_schema(path):
    p = Path(path)

    result = {
        "test": "database_schema_has_readings_table",
        "database_path": str(p),
        "exists": p.exists(),
        "has_readings_table": False,
        "tables_found": [],
        "error": None
    }

    if not p.exists():
        return result

    try:
        conn = sqlite3.connect(p)

        tables = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()

        conn.close()

        names = [row[0] for row in tables]

        result["tables_found"] = names
        result["has_readings_table"] = "readings" in names

    except Exception as exc:
        result["error"] = f"{type(exc).__name__}: {exc}"

    return result

cause2_before = verify_database_schema(DB_FILE)

print(json.dumps(cause2_before, indent=2))


{
  "test": "database_schema_has_readings_table",
  "database_path": "/content/module7_assignment7_2_instructor_demo/module7_demo.db",
  "exists": true,
  "has_readings_table": false,
  "tables_found": [],
  "error": null
}


## 9. Recovery Procedure

Restore the original database, then verify:
- expected path;
- schema;
- preserved rows;
- final health state.


In [10]:
# SQLite may have created a blank database at the expected path.
if DB_FILE.exists():
    DB_FILE.unlink()

shutil.move(OFFLINE_DB, DB_FILE)

log_event(
    "INFO",
    "recovery_action",
    "storage",
    action="restored original database file to expected path",
    database_path=str(DB_FILE),
    health_state="RECOVERING"
)

cause1_after = verify_database_path(DB_FILE)
cause2_after = verify_database_schema(DB_FILE)

conn = sqlite3.connect(DB_FILE)

rows_preserved = conn.execute(
    "SELECT COUNT(*) FROM readings"
).fetchone()[0]

conn.close()

recovery_passed = (
    cause1_after["exists"]
    and cause2_after["has_readings_table"]
    and rows_preserved == 20
)

log_event(
    "INFO",
    "recovery_verified",
    "system",
    database_available=cause1_after["exists"],
    schema_valid=cause2_after["has_readings_table"],
    rows_preserved=rows_preserved,
    recovery_passed=recovery_passed,
    health_state="HEALTHY" if recovery_passed else "FAILED"
)

print("Recovery passed:", recovery_passed)
print("Rows preserved:", rows_preserved)

assert recovery_passed


Recovery passed: True
Rows preserved: 20


## 10. What the Logs Told Me / Did Not Tell Me

A strong failure report includes both.

### The logs told us
- the system was healthy before the failure;
- the expected database became unavailable;
- access to the required table failed;
- recovery was attempted;
- recovery verification passed.

### The logs did not prove
- who or what changed the path in a real incident;
- whether permissions changed;
- whether the disk was full;
- whether another OS/service event occurred first;
- the ultimate root cause outside the application's own evidence.


In [11]:
what_logs_told = [
    "System had a healthy baseline.",
    "Database access failed after the controlled failure.",
    "Expected application schema was unavailable at the expected path.",
    "Recovery restored path, schema, and original row count."
]

what_logs_did_not_tell = [
    "Who or what changed the database path in a real incident.",
    "Whether disk pressure contributed.",
    "Whether permissions changed.",
    "Whether an external OS/service event occurred first."
]

print("WHAT THE LOGS TOLD ME")
for item in what_logs_told:
    print(" -", item)

print("\nWHAT THE LOGS DID NOT TELL ME")
for item in what_logs_did_not_tell:
    print(" -", item)


WHAT THE LOGS TOLD ME
 - System had a healthy baseline.
 - Database access failed after the controlled failure.
 - Expected application schema was unavailable at the expected path.
 - Recovery restored path, schema, and original row count.

WHAT THE LOGS DID NOT TELL ME
 - Who or what changed the database path in a real incident.
 - Whether disk pressure contributed.
 - Whether permissions changed.
 - Whether an external OS/service event occurred first.


## 11. Improvement Made

The investigation should improve the system.

This demo adds a log entry documenting the fields that should be captured next time.


In [12]:
log_event(
    "INFO",
    "logging_improvement_added",
    "logging",
    improvement=(
        "Future storage failures should include component, database_path, "
        "exception_type, health_state, attempted_recovery, and verification_result."
    ),
    health_state="HEALTHY"
)

print("Logging improvement recorded.")


Logging improvement recorded.


# 12. Ollama — Evidence Review and Red Team

Ollama is used **after** deterministic evidence is established.

### Ollama may
- summarize log evidence;
- identify plausible causes;
- identify missing evidence;
- propose verification tests;
- challenge a conclusion;
- help draft the report.

### Ollama may not
- invent log events;
- invent sensor readings;
- change deterministic test results;
- claim an unverified root cause is proven.


In [13]:
# Install and start Ollama in Colab
import subprocess
import shutil as _shutil
import time as _time
import requests

def run_shell(cmd):
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout[-2500:])

    if result.returncode != 0:
        print(result.stderr[-2500:])

    return result.returncode

run_shell("apt-get update -qq")
run_shell("apt-get install -y -qq curl zstd")

if _shutil.which("ollama") is None:
    run_shell("curl -fsSL https://ollama.com/install.sh | sh")

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "qwen2.5:0.5b"

def ollama_running():
    try:
        return requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=3
        ).status_code == 200
    except requests.RequestException:
        return False

if not ollama_running():
    log = open("/tmp/ollama_module7.log", "a")

    proc = subprocess.Popen(
        ["ollama","serve"],
        stdout=log,
        stderr=subprocess.STDOUT
    )

    for _ in range(30):
        if ollama_running():
            break
        _time.sleep(1)

if not ollama_running():
    raise RuntimeError("Ollama did not start.")

run_shell(f"ollama pull {OLLAMA_MODEL}")

print("Ollama ready:", OLLAMA_MODEL)



(Reading database ... 70%
(Reading database ... 75%
(Reading database ... 80%
(Reading database ... 85%
(Reading database ... 90%
(Reading database ... 95%
(Reading database ... 100%
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../libcurl4-openssl-dev_8.5.0-2ubuntu10.16_amd64.deb ...
Unpacking libcurl4-openssl-dev:amd64 (8.5.0-2ubuntu10.16) over (8.5.0-2ubuntu10.15) ...
Preparing to unpack .../curl_8.5.0-2ubuntu10.16_amd64.deb ...
Unpacking curl (8.5.0-2ubuntu10.16) over (8.5.0-2ubuntu10.15) ...
Preparing to unpack .../libcurl4t64_8.5.0-2ubuntu10.16_amd64.deb ...
Unpacking libcurl4t64:amd64 (8.5.0-2ubuntu10.16) over (8.5.0-2ubuntu10.15) ...
Preparing to unpack .../libcurl3t64-gnutls_8.5.0-2ubuntu10.16_amd64.deb ...
Unpacking libcurl3t64-gnutls:amd64 (8.5.0-2ubuntu10.16) over (8.5.0-2ubuntu10.15) ...
Selecting previously unselected package zstd.
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-

In [14]:
AI_SYSTEM_POLICY = '''
You are an engineering-software incident-review assistant for EGN 321.

Use only evidence supplied in the prompt.

Rules:
1. Do not invent events, timestamps, readings, files, causes, or test results.
2. Separate observation, inference, and hypothesis.
3. Never claim a root cause is proven unless deterministic evidence proves it.
4. For every possible cause include:
   - supporting evidence,
   - missing evidence,
   - deterministic verification test.
5. Challenge conclusions that go beyond the evidence.
6. If evidence is insufficient, say exactly what evidence is missing.
'''

def ask_ollama(prompt, timeout=600, num_predict=700):
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "system": AI_SYSTEM_POLICY,
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0.2,
            "num_predict": num_predict,
            "num_ctx": 4096
        }
    }

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=payload,
        timeout=(10, timeout)
    )

    response.raise_for_status()
    return response.json()["response"].strip()

print("Warm-up:", ask_ollama("Reply with exactly READY.", num_predict=10))


Warm-up: READY


## 13. Build the Deterministic Evidence Package for Ollama


In [15]:
# Re-read final logs after recovery and improvement.
records = []

with open(LOG_FILE, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

logs = pd.DataFrame(records)
logs["timestamp"] = pd.to_datetime(logs["timestamp"], utc=True)
logs = logs.sort_values("timestamp").reset_index(drop=True)

evidence = {
    "first_observable_symptom": FIRST_OBSERVABLE,
    "cause_1_test_before_recovery": cause1_before,
    "cause_2_test_before_recovery": cause2_before,
    "cause_1_test_after_recovery": cause1_after,
    "cause_2_test_after_recovery": cause2_after,
    "rows_preserved_after_recovery": int(rows_preserved),
    "recovery_passed": bool(recovery_passed),
    "what_logs_told": what_logs_told,
    "what_logs_did_not_tell": what_logs_did_not_tell,
    "event_counts": logs["event"].value_counts().to_dict()
}

print(json.dumps(evidence, indent=2))


{
  "first_observable_symptom": "failure_injected",
  "cause_1_test_before_recovery": {
    "test": "database_path_exists",
    "database_path": "/content/module7_assignment7_2_instructor_demo/module7_demo.db",
    "exists": true
  },
  "cause_2_test_before_recovery": {
    "test": "database_schema_has_readings_table",
    "database_path": "/content/module7_assignment7_2_instructor_demo/module7_demo.db",
    "exists": true,
    "has_readings_table": false,
    "tables_found": [],
    "error": null
  },
  "cause_1_test_after_recovery": {
    "test": "database_path_exists",
    "database_path": "/content/module7_assignment7_2_instructor_demo/module7_demo.db",
    "exists": true
  },
  "cause_2_test_after_recovery": {
    "test": "database_schema_has_readings_table",
    "database_path": "/content/module7_assignment7_2_instructor_demo/module7_demo.db",
    "exists": true,
    "has_readings_table": true,
    "tables_found": [
      "readings",
      "sqlite_sequence"
    ],
    "error": nu

## 14. Ollama Investigator Review


In [16]:
investigator_prompt = f'''
Here is deterministic incident evidence:

{json.dumps(evidence, indent=2)}

Return exactly these sections:

1. What the evidence proves
2. What the evidence does NOT prove
3. Possible Cause 1
   - supporting evidence
   - missing evidence
   - deterministic verification test
4. Possible Cause 2
   - supporting evidence
   - missing evidence
   - deterministic verification test
5. Recommended logging improvement

Do not invent events.
Do not call a hypothesis proven unless the deterministic evidence proves it.
'''

ollama_investigation = ask_ollama(
    investigator_prompt,
    num_predict=900
)

print(ollama_investigation)


1. What the evidence proves:
   - The evidence proves that the database path was successfully accessed and the schema was read before the failure.
   - It also proves that the recovery process restored the schema and the original row count.

2. What the evidence does NOT prove:
   - The evidence does not prove that the application schema was unavailable at the expected path or that disk pressure contributed to the failure.
   - It does not prove that the recovery action was the expected one or that the external OS/service event occurred first.

3. Possible Cause 1:
   - The evidence does not provide any evidence to prove that the application schema was unavailable at the expected path or that disk pressure contributed to the failure.

4. Possible Cause 2:
   - The evidence does not provide any evidence to prove that the recovery action was the expected one or that the external OS/service event occurred first.

5. Recommended logging improvement:
   - The evidence does not provide any e

## 15. Ollama Red-Team Review

Now ask Ollama to challenge a conclusion that sounds reasonable but may overstate the evidence.


In [17]:
provisional_conclusion = (
    "The database failure was caused by a corrupted database file."
)

red_team_prompt = f'''
Deterministic evidence:

{json.dumps(evidence, indent=2)}

Provisional conclusion:
{provisional_conclusion}

Red-team this conclusion.

Return:
- what the evidence actually supports;
- what the conclusion overstates;
- alternative explanations;
- the strongest next deterministic test.

Do not invent new evidence.
'''

ollama_red_team = ask_ollama(
    red_team_prompt,
    num_predict=700
)

print(ollama_red_team)


- what the evidence actually supports: The database failure was caused by a corrupted database file.
- what the conclusion overstates: The database failure was caused by a corrupted database file.
- alternative explanations: The database failure was caused by a corrupted database file, but it is not clear whether it was caused by a corrupted database file or a corrupted file system.
- the strongest next deterministic test: The database failure was caused by a corrupted database file.


## 16. Generate the Required Evidence Table Again After Recovery


In [18]:
def interpretation_for_event(row):
    event = row["event"]

    mapping = {
        "system_baseline": "System confirmed healthy before failure.",
        "failure_injected": "Expected database path intentionally made unavailable.",
        "database_read_failed": "Application could not access required database schema.",
        "recovery_action": "Original database restored to expected path.",
        "recovery_verified": "Path, schema, row count, and health verified after recovery.",
        "logging_improvement_added": "Logging requirements improved after investigation."
    }

    return mapping.get(
        event,
        "Operational event; interpretation requires surrounding evidence."
    )

required_evidence = pd.DataFrame({
    "Time": logs["timestamp"].dt.strftime("%H:%M:%S"),
    "Component": logs["component"],
    "Event": logs["event"],
    "Evidence": logs["level"].map(lambda x: f"{x} log entry"),
    "Interpretation": logs.apply(interpretation_for_event, axis=1)
})

display(required_evidence.tail(12))


,Time,Component,Event,Evidence,Interpretation
14,20:09:05,sensor,reading_stored,INFO log entry,Operational event; interpretation requires sur...
15,20:09:05,sensor,reading_stored,INFO log entry,Operational event; interpretation requires sur...
16,20:09:05,sensor,reading_stored,INFO log entry,Operational event; interpretation requires sur...
17,20:09:05,sensor,reading_stored,INFO log entry,Operational event; interpretation requires sur...
18,20:09:05,sensor,reading_stored,INFO log entry,Operational event; interpretation requires sur...
19,20:09:05,sensor,reading_stored,INFO log entry,Operational event; interpretation requires sur...
20,20:09:05,system,system_baseline,INFO log entry,System confirmed healthy before failure.
21,20:09:17,storage,failure_injected,WARNING log entry,Expected database path intentionally made unav...
22,20:09:17,storage,database_read_failed,ERROR log entry,Application could not access required database...
23,20:10:32,storage,recovery_action,INFO log entry,Original database restored to expected path.


# 17. Generate the Full Failure Report

The Word report uses the exact required headings from Assignment 7.2.


In [22]:
# ============================================================
# STEP 17 - GENERATE THE FULL FAILURE REPORT
# COLAB-SAFE VERSION
# ============================================================

# Install python-docx if needed
try:
    from docx import Document
    from docx.shared import Pt
except ModuleNotFoundError:
    print("Installing python-docx...")
    %pip install -q python-docx
    from docx import Document
    from docx.shared import Pt

print("✅ python-docx ready.")


# ------------------------------------------------------------
# Report output file
# ------------------------------------------------------------

REPORT_FILE = (
    BASE
    / "EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx"
)


# ------------------------------------------------------------
# Create Word document
# ------------------------------------------------------------

doc = Document()

doc.styles["Normal"].font.name = "Aptos"
doc.styles["Normal"].font.size = Pt(10.5)


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

doc.add_heading(
    "EGN 321 — Module 7 Assignment 7.2",
    0
)

doc.add_paragraph(
    "Failure Report: Diagnose, Recover, and Improve the System"
)


# ------------------------------------------------------------
# Required Assignment Sections
# ------------------------------------------------------------

sections = [

    (
        "1. System Before Failure",

        "The application was operating in a HEALTHY state. "
        "The SQLite database existed, the expected readings table "
        "was available, and 20 valid readings had been stored."
    ),

    (
        "2. Failure Introduced",

        "The database file was deliberately renamed to simulate "
        "a realistic database-unavailable or incorrect-path failure."
    ),

    (
        "3. First Observable Symptom",

        f"The first observable warning/error in the available logs was "
        f"'{FIRST_OBSERVABLE}'. "
        f"This is an observed symptom and is not automatically "
        f"the final root cause."
    ),

    (
        "4. Log Evidence",

        "Structured logs recorded the healthy baseline, controlled "
        "failure, database access failure, recovery action, recovery "
        "verification, and logging improvement."
    ),

    (
        "5. Timeline of Events",

        "Healthy baseline → failure injected → database access failed → "
        "recovery action → recovery verified → logging improved."
    ),

    (
        "6. Possible Cause 1",

        f"Possible cause: the expected database path became unavailable. "
        f"Deterministic path test before recovery: {cause1_before}. "
        "A missing expected path supports this hypothesis."
    ),

    (
        "7. Possible Cause 2",

        f"Possible cause: a database file exists but the required "
        f"application schema is missing or invalid. "
        f"Deterministic schema test before recovery: {cause2_before}. "
        "In this controlled demo, the expected application database "
        "was unavailable."
    ),

    (
        "8. Recovery Procedure",

        "Remove any blank SQLite file created at the expected path, "
        "restore the original database file, verify the expected path, "
        "verify the readings table, and confirm preserved data."
    ),

    (
        "9. Verification of Recovery",

        f"Recovery passed: {recovery_passed}. "
        f"Rows preserved: {rows_preserved}. "
        f"Path after recovery: {cause1_after}. "
        f"Schema after recovery: {cause2_after}."
    ),

    (
        "10. What the Logs Told Me",

        "The logs established the sequence of events and confirmed "
        "that database access failed after the controlled failure. "
        "They also confirmed that recovery was attempted and that "
        "final verification passed."
    ),

    (
        "11. What the Logs Did Not Tell Me",

        "The application logs alone would not prove who or what changed "
        "the database path in a real incident. They also do not prove "
        "whether disk pressure, permissions, or another operating-system "
        "event contributed."
    ),

    (
        "12. Improvement Made",

        "Logging was improved to require component, database path, "
        "exception type, health state, attempted recovery, and "
        "verification result for future storage failures."
    ),

    (
        "13. Remaining Limitations",

        "This is a controlled classroom incident. A real deployment "
        "would also require operating-system, service-manager, storage, "
        "permission, and deployment-platform evidence."
    )
]


# ------------------------------------------------------------
# Add report sections
# ------------------------------------------------------------

for heading, body in sections:

    doc.add_heading(
        heading,
        level=1
    )

    doc.add_paragraph(
        body
    )


# ------------------------------------------------------------
# Required Evidence Table
# ------------------------------------------------------------

doc.add_heading(
    "Required Evidence Table",
    level=1
)

table = doc.add_table(
    rows=1,
    cols=5
)

table.style = "Table Grid"


headers = [
    "Time",
    "Component",
    "Event",
    "Evidence",
    "Interpretation"
]


# Header row
for i, heading in enumerate(headers):

    table.rows[0].cells[i].text = heading


# Add evidence rows
for _, row in required_evidence.tail(12).iterrows():

    cells = table.add_row().cells

    for i, heading in enumerate(headers):

        cells[i].text = str(
            row[heading]
        )


# ------------------------------------------------------------
# Ollama Review Section
# ------------------------------------------------------------

doc.add_heading(
    "Ollama Review",
    level=1
)

doc.add_paragraph(
    "Ollama was used after deterministic evidence had been established. "
    "Its role was to review hypotheses, identify missing evidence, "
    "propose verification tests, and challenge conclusions. "
    "AI output was not treated as operational evidence."
)


# Investigator Review
doc.add_heading(
    "Ollama Investigator Review",
    level=2
)

doc.add_paragraph(
    ollama_investigation
)


# Red-Team Review
doc.add_heading(
    "Ollama Red-Team Review",
    level=2
)

doc.add_paragraph(
    ollama_red_team
)


# ------------------------------------------------------------
# Save Report
# ------------------------------------------------------------

doc.save(
    REPORT_FILE
)


# ------------------------------------------------------------
# Verify Report Was Created
# ------------------------------------------------------------

if REPORT_FILE.exists():

    size_kb = (
        REPORT_FILE.stat().st_size
        / 1024
    )

    print("\n" + "=" * 60)
    print("✅ FAILURE REPORT CREATED SUCCESSFULLY")
    print("=" * 60)

    print(
        "File:",
        REPORT_FILE.name
    )

    print(
        "Size:",
        f"{size_kb:.1f} KB"
    )

    print(
        "Location:",
        REPORT_FILE
    )

else:

    raise FileNotFoundError(
        "The Word report was not created."
    )


# ------------------------------------------------------------
# Optional Download
# ------------------------------------------------------------

# Uncomment these two lines if you want Colab
# to immediately download the Word report.

# from google.colab import files
# files.download(str(REPORT_FILE))


Installing python-docx...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 2.8 MB/s eta 0:00:00
✅ python-docx ready.

✅ FAILURE REPORT CREATED SUCCESSFULLY
File: EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx
Size: 38.1 KB
Location: /content/module7_assignment7_2_instructor_demo/EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx


# 18. Generate README.md and AI_LOG.md


In [23]:
README_FILE = BASE / "README.md"
AI_LOG_FILE = BASE / "AI_LOG.md"

README_FILE.write_text(
f'''# EGN 321 Module 7 Assignment 7.2 Demo

## Failure Scenario
Controlled database-unavailable condition.

## Investigation Flow
Healthy baseline -> controlled failure -> structured logs -> evidence table ->
first observable symptom -> two possible causes -> deterministic tests ->
recovery -> verification -> logging improvement -> Ollama review.

## Recovery Result
Recovery passed: {recovery_passed}
Rows preserved: {rows_preserved}

## Required Evidence
- structured logs
- failure timeline
- failed-state evidence
- recovered-state evidence
- deterministic verification tests
- failure report
- updated README
- AI_LOG.md

## Important
Logs show what the application observed.
They do not automatically prove the ultimate root cause.

Ollama is a reviewer and drafting assistant.
It is not the source of operational evidence.
''',
encoding="utf-8"
)

AI_LOG_FILE.write_text(
f'''# AI_LOG.md

Tool: Ollama
Model: {OLLAMA_MODEL}

## Purpose
Review deterministic incident evidence, propose hypotheses and tests,
red-team conclusions, and assist with failure-report drafting.

## Evidence Provided
{json.dumps(evidence, indent=2)}

## Investigator Review
{ollama_investigation}

## Red-Team Review
{ollama_red_team}

## Verification Rule
AI output is not treated as evidence.
Claims must be verified using logs, SQLite, observed system behavior,
or deterministic tests.

## Instructor / Student Review
Document:
- which AI suggestions were accepted;
- which were rejected;
- which deterministic test was added;
- what changed in the final report.
''',
encoding="utf-8"
)

print("Created:", README_FILE)
print("Created:", AI_LOG_FILE)


Created: /content/module7_assignment7_2_instructor_demo/README.md
Created: /content/module7_assignment7_2_instructor_demo/AI_LOG.md


# 19. Export All Required Assignment Deliverables

This creates a single ZIP containing:

- failure report;
- relevant log files;
- failure timeline;
- failed-condition evidence;
- recovered-condition evidence;
- updated application/evidence code;
- updated README;
- verification tests/results;
- `AI_LOG.md`;
- Ollama investigator review;
- Ollama red-team review;
- required evidence table.


In [24]:
# ============================================================
# FINAL EXPORT STEP
# MODULE 7 ASSIGNMENT 7.2
# COLAB-SAFE / SELF-CONTAINED VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import json
import shutil
import zipfile


# ------------------------------------------------------------
# 1. Re-establish important file paths
# ------------------------------------------------------------

# BASE should already exist from the beginning of the notebook.
if "BASE" not in globals():
    BASE = Path("/content/module7_assignment7_2_instructor_demo")

BASE = Path(BASE)

EXPORT_DIR = BASE / "assignment7_2_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_FILE = (
    BASE
    / "EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx"
)

README_FILE = BASE / "README.md"
AI_LOG_FILE = BASE / "AI_LOG.md"

ZIP_FILE = Path(
    "/content/EGN321_Module7_Assignment7_2_COMPLETE_DEMO_EXPORT.zip"
)

print("BASE:", BASE)
print("EXPORT_DIR:", EXPORT_DIR)


# ------------------------------------------------------------
# 2. Verify required runtime variables
# ------------------------------------------------------------

required_variables = [
    "LOG_FILE",
    "DB_FILE",
    "cause1_before",
    "cause2_before",
    "cause1_after",
    "cause2_after",
    "rows_preserved",
    "recovery_passed",
    "FIRST_OBSERVABLE",
    "required_evidence"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "The following required notebook variables are missing:\n"
        + "\n".join(
            f" - {name}"
            for name in missing_variables
        )
        + "\n\nRun the earlier investigation/recovery cells first."
    )


# ------------------------------------------------------------
# 3. Re-read final structured logs
# ------------------------------------------------------------

records = []

with open(
    LOG_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:

        line = line.strip()

        if line:
            records.append(
                json.loads(line)
            )


logs = pd.DataFrame(records)

if logs.empty:
    raise RuntimeError(
        "The log file was read successfully, "
        "but it contained no log records."
    )


logs["timestamp"] = pd.to_datetime(
    logs["timestamp"],
    errors="coerce",
    utc=True
)

logs = (
    logs
    .sort_values("timestamp")
    .reset_index(drop=True)
)


print(
    "Final log records:",
    len(logs)
)


# ------------------------------------------------------------
# 4. Rebuild failure timeline
# ------------------------------------------------------------

timeline_events = [

    "system_baseline",

    "failure_injected",

    "database_read_failed",

    "recovery_action",

    "recovery_verified",

    "logging_improvement_added"
]


failure_timeline = (
    logs[
        logs["event"]
        .isin(timeline_events)
    ]
    .copy()
)


print(
    "Timeline events:",
    len(failure_timeline)
)


# ------------------------------------------------------------
# 5. Export log evidence
# ------------------------------------------------------------

logs.to_csv(
    EXPORT_DIR
    / "relevant_log_records.csv",
    index=False
)


failure_timeline.to_csv(
    EXPORT_DIR
    / "failure_timeline.csv",
    index=False
)


required_evidence.to_csv(
    EXPORT_DIR
    / "required_evidence_table.csv",
    index=False
)


# ------------------------------------------------------------
# 6. Verification results
# ------------------------------------------------------------

verification_results = {

    "possible_cause_1_before":
        cause1_before,

    "possible_cause_2_before":
        cause2_before,

    "possible_cause_1_after":
        cause1_after,

    "possible_cause_2_after":
        cause2_after,

    "rows_preserved":
        int(rows_preserved),

    "recovery_passed":
        bool(recovery_passed)
}


(
    EXPORT_DIR
    / "verification_results.json"
).write_text(

    json.dumps(
        verification_results,
        indent=2
    ),

    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. Failed-condition evidence
# ------------------------------------------------------------

failed_condition_text = f"""
MODULE 7 ASSIGNMENT 7.2
FAILED CONDITION EVIDENCE

First observable symptom:
{FIRST_OBSERVABLE}


POSSIBLE CAUSE 1
Expected database path became unavailable.

Deterministic test before recovery:

{json.dumps(cause1_before, indent=2)}


POSSIBLE CAUSE 2
Database/schema may be unavailable or invalid.

Deterministic test before recovery:

{json.dumps(cause2_before, indent=2)}


IMPORTANT:
These results are evidence.

They do not automatically prove the final root cause.
""".strip()


(
    EXPORT_DIR
    / "failed_condition_evidence.txt"
).write_text(

    failed_condition_text,

    encoding="utf-8"
)


# ------------------------------------------------------------
# 8. Recovered-condition evidence
# ------------------------------------------------------------

recovered_condition_text = f"""
MODULE 7 ASSIGNMENT 7.2
RECOVERED CONDITION EVIDENCE

Recovery passed:
{recovery_passed}

Rows preserved:
{rows_preserved}


PATH TEST AFTER RECOVERY

{json.dumps(cause1_after, indent=2)}


SCHEMA TEST AFTER RECOVERY

{json.dumps(cause2_after, indent=2)}


INTERPRETATION

Recovery is considered successful only because:
- the expected database path exists;
- the expected schema exists;
- the original rows were preserved;
- the deterministic recovery test passed.
""".strip()


(
    EXPORT_DIR
    / "recovered_condition_evidence.txt"
).write_text(

    recovered_condition_text,

    encoding="utf-8"
)


# ------------------------------------------------------------
# 9. Export Ollama review
# ------------------------------------------------------------

if "ollama_investigation" in globals():

    (
        EXPORT_DIR
        / "ollama_investigator_review.txt"
    ).write_text(

        str(
            ollama_investigation
        ),

        encoding="utf-8"
    )

else:

    (
        EXPORT_DIR
        / "ollama_investigator_review.txt"
    ).write_text(

        "Ollama investigator review was not available "
        "in the current runtime.",

        encoding="utf-8"
    )


if "ollama_red_team" in globals():

    (
        EXPORT_DIR
        / "ollama_red_team_review.txt"
    ).write_text(

        str(
            ollama_red_team
        ),

        encoding="utf-8"
    )

else:

    (
        EXPORT_DIR
        / "ollama_red_team_review.txt"
    ).write_text(

        "Ollama red-team review was not available "
        "in the current runtime.",

        encoding="utf-8"
    )


# ------------------------------------------------------------
# 10. Copy Word failure report
# ------------------------------------------------------------

if REPORT_FILE.exists():

    shutil.copy2(
        REPORT_FILE,
        EXPORT_DIR
        / REPORT_FILE.name
    )

    print(
        "✅ Added report:",
        REPORT_FILE.name
    )

else:

    print(
        "⚠️ Word report not found:"
    )

    print(
        REPORT_FILE
    )

    print(
        "Run Step 17 before exporting "
        "if you want the Word report included."
    )


# ------------------------------------------------------------
# 11. Copy original structured log
# ------------------------------------------------------------

LOG_FILE = Path(LOG_FILE)

if LOG_FILE.exists():

    shutil.copy2(
        LOG_FILE,
        EXPORT_DIR
        / "module7_app.jsonl"
    )

    print(
        "✅ Added structured log"
    )

else:

    print(
        "⚠️ Structured log file not found:",
        LOG_FILE
    )


# ------------------------------------------------------------
# 12. Copy recovered database
# ------------------------------------------------------------

DB_FILE = Path(DB_FILE)

if DB_FILE.exists():

    shutil.copy2(
        DB_FILE,
        EXPORT_DIR
        / "recovered_database.db"
    )

    print(
        "✅ Added recovered database"
    )

else:

    print(
        "⚠️ Recovered database not found:",
        DB_FILE
    )


# ------------------------------------------------------------
# 13. Copy README
# ------------------------------------------------------------

if README_FILE.exists():

    shutil.copy2(
        README_FILE,
        EXPORT_DIR
        / "README.md"
    )

    print(
        "✅ Added README.md"
    )

else:

    print(
        "⚠️ README.md not found."
    )

    print(
        "Run the README generation step "
        "before export if needed."
    )


# ------------------------------------------------------------
# 14. Copy AI_LOG
# ------------------------------------------------------------

if AI_LOG_FILE.exists():

    shutil.copy2(
        AI_LOG_FILE,
        EXPORT_DIR
        / "AI_LOG.md"
    )

    print(
        "✅ Added AI_LOG.md"
    )

else:

    print(
        "⚠️ AI_LOG.md not found."
    )

    print(
        "Run the AI_LOG generation step "
        "before export if needed."
    )


# ------------------------------------------------------------
# 15. Create submission manifest
# ------------------------------------------------------------

SUBMISSION_MANIFEST = (
    EXPORT_DIR
    / "SUBMISSION_MANIFEST.txt"
)


manifest = """
MODULE 7 ASSIGNMENT 7.2
DEMO SUBMISSION PACKAGE


[1] FAILURE REPORT

EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx


[2] RELEVANT LOGS

module7_app.jsonl

relevant_log_records.csv


[3] FAILURE TIMELINE

failure_timeline.csv


[4] FAILED CONDITION EVIDENCE

failed_condition_evidence.txt


[5] RECOVERED CONDITION EVIDENCE

recovered_condition_evidence.txt

recovered_database.db


[6] VERIFICATION TESTS / RESULTS

verification_results.json


[7] REQUIRED EVIDENCE TABLE

required_evidence_table.csv


[8] UPDATED README

README.md


[9] AI LOG

AI_LOG.md


[10] OLLAMA SUPPORTING REVIEW

ollama_investigator_review.txt

ollama_red_team_review.txt


[11] GITHUB

Student submission should include the GitHub repository link
in the LMS submission.


IMPORTANT COURSE RULE

Logs and deterministic tests establish the evidence.

Ollama may review, challenge, summarize, and propose tests.

Ollama is not operational evidence.
""".strip()


SUBMISSION_MANIFEST.write_text(
    manifest,
    encoding="utf-8"
)


print(
    "✅ Created submission manifest"
)


# ------------------------------------------------------------
# 16. Show export folder before ZIP creation
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 65
)

print(
    "FILES READY FOR EXPORT"
)

print(
    "=" * 65
)


for p in sorted(
    EXPORT_DIR.iterdir()
):

    if p.is_file():

        size_kb = (
            p.stat().st_size
            / 1024
        )

        print(
            f"✅ {p.name} "
            f"({size_kb:.1f} KB)"
        )


# ------------------------------------------------------------
# 17. Create final ZIP
# ------------------------------------------------------------

if ZIP_FILE.exists():
    ZIP_FILE.unlink()


with zipfile.ZipFile(
    ZIP_FILE,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for p in sorted(
        EXPORT_DIR.iterdir()
    ):

        if p.is_file():

            z.write(
                p,
                arcname=p.name
            )


# ------------------------------------------------------------
# 18. Verify ZIP
# ------------------------------------------------------------

if not ZIP_FILE.exists():

    raise FileNotFoundError(
        "The final ZIP was not created."
    )


zip_size_mb = (
    ZIP_FILE.stat().st_size
    / 1024
    / 1024
)


print(
    "\n"
    + "=" * 65
)

print(
    "✅ ASSIGNMENT 7.2 EXPORT COMPLETE"
)

print(
    "=" * 65
)


print(
    "ZIP:",
    ZIP_FILE.name
)

print(
    "Size:",
    f"{zip_size_mb:.2f} MB"
)

print(
    "Location:",
    ZIP_FILE
)


# ------------------------------------------------------------
# 19. List files inside ZIP
# ------------------------------------------------------------

print(
    "\nFiles inside ZIP:"
)


with zipfile.ZipFile(
    ZIP_FILE,
    "r"
) as z:

    for name in z.namelist():

        print(
            " -",
            name
        )


# ------------------------------------------------------------
# 20. Optional automatic download
# ------------------------------------------------------------

# Uncomment these lines to download immediately.

# from google.colab import files
# files.download(str(ZIP_FILE))

BASE: /content/module7_assignment7_2_instructor_demo
EXPORT_DIR: /content/module7_assignment7_2_instructor_demo/assignment7_2_export
Final log records: 26
Timeline events: 6
✅ Added report: EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx
✅ Added structured log
✅ Added recovered database
✅ Added README.md
✅ Added AI_LOG.md
✅ Created submission manifest

FILES READY FOR EXPORT
✅ AI_LOG.md (7.6 KB)
✅ EGN321_Module7_Assignment7_2_Failure_Report_Demo.docx (38.1 KB)
✅ README.md (0.8 KB)
✅ SUBMISSION_MANIFEST.txt (0.9 KB)
✅ failed_condition_evidence.txt (0.7 KB)
✅ failure_timeline.csv (1.3 KB)
✅ module7_app.jsonl (5.8 KB)
✅ ollama_investigator_review.txt (4.8 KB)
✅ ollama_red_team_review.txt (0.5 KB)
✅ recovered_condition_evidence.txt (0.8 KB)
✅ recovered_database.db (12.0 KB)
✅ relevant_log_records.csv (3.1 KB)
✅ required_evidence_table.csv (2.8 KB)
✅ verification_results.json (1.0 KB)

✅ ASSIGNMENT 7.2 EXPORT COMPLETE
ZIP: EGN321_Module7_Assignment7_2_COMPLETE_DEMO_EXPORT.zip
Size: 0.

# Demo Complete

The demo now covers the full Assignment 7.2 expectation:

- exact 13 report sections;
- required evidence table;
- failure report;
- relevant logs;
- failure timeline;
- failed-state evidence;
- recovered-state evidence;
- verification tests;
- README;
- AI_LOG.md;
- Ollama evidence review;
- Ollama red-team review;
- final submission ZIP.

### Instructor closing point

> The strongest report is not the one with the most confident root-cause statement.  
> It is the one that clearly separates **evidence, hypothesis, missing evidence, verification, and remaining uncertainty**.
